# 03. Prompt Management

In this notebook, we will learn how to centrally manage prompts
outside of code using Langfuse's prompt management features.

## What you will learn
- Creating prompts (`create_prompt()`)
- Retrieving prompts and substituting variables (`get_prompt()`, `compile()`)
- Version management and labels
- Linking traces to prompts

## Why should you manage prompts centrally?

| Approach | Problems |
|----------|----------|
| Hardcoded in code | Requires deploy for changes, hard to track versions |
| Managed with Langfuse | Reflects changes instantly without deploy, maintains version history, enables team collaboration |

**Langfuse prompt structure:**
- **Name**: Prompt identifier
- **Template**: Template text using `{{variable}}` format
- **Config**: Stores settings such as model, temperature, etc.
- **Labels**: Distinguishes environments like `production`, `staging`, etc.
- **Version**: Automatically assigns version numbers

## Step 1: Initial setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["ANTHROPIC_API_KEY"] = os.environ["CLAUDE_API_KEY"]

from langfuse import get_client, observe
from opentelemetry.instrumentation.anthropic import AnthropicInstrumentor
from anthropic import Anthropic

AnthropicInstrumentor().instrument()
langfuse = get_client()
client = Anthropic()

print("✅ Setup complete")

## Step 2: Create a prompt (Text type)

Create a basic text prompt.
Use the `{{variable_name}}` format to specify dynamic variables.

In [ ]:
langfuse.create_prompt(
    name="text-translator",
    prompt="""Please translate the following text into {{target_language}}.
Preserve the nuance and formality level of the original.

Source text:
{{source_text}}

Translation:""",
    config={
        "model": "claude-sonnet-4-6",
        "max_tokens": 1024,
        "temperature": 0.3
    },
    labels=["production"]
)

print("✅ 'text-translator' prompt created")

## Step 3: Create a prompt (Chat type)

You can also create chat-format prompts with system/user/assistant roles.

In [ ]:
langfuse.create_prompt(
    name="code-reviewer",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": """You are an expert {{language}} code reviewer.
Analyze the code and review the following items:
1. Bugs and potential errors
2. Code quality and readability
3. Performance improvements
4. Security vulnerabilities

Provide specific feedback and improvement suggestions for each item."""
        },
        {
            "role": "user",
            "content": "Please review the following code:\n\n```{{language}}\n{{code}}\n```"
        }
    ],
    config={
        "model": "claude-sonnet-4-6",
        "max_tokens": 2048
    },
    labels=["production"]
)

print("✅ 'code-reviewer' prompt created")

> **Check on the dashboard:**
> [https://cloud.langfuse.com](https://cloud.langfuse.com) → **Prompt Management**
> You should see the two prompts you just created.

## Step 4: Retrieve a prompt and substitute variables

In [ ]:
# Fetch the latest version with the production label
translator_prompt = langfuse.get_prompt("text-translator")

print(f"Prompt name: {translator_prompt.name}")
print(f"Version: {translator_prompt.version}")
print(f"Labels: {translator_prompt.labels}")
print(f"\nTemplate:\n{translator_prompt.prompt}")

In [ ]:
# Use compile() to substitute variables with actual values
compiled_prompt = translator_prompt.compile(
    target_language="English",
    source_text="Artificial intelligence is greatly changing our daily lives."
)

print("Compiled prompt:")
print(compiled_prompt)

## Step 5: Call Claude using the prompt

In [ ]:
@observe()
def translate(source_text: str, target_language: str) -> str:
    prompt_template = langfuse.get_prompt("text-translator")
    compiled = prompt_template.compile(
        source_text=source_text,
        target_language=target_language
    )
    
    config = prompt_template.config
    
    response = client.messages.create(
        model=config.get("model", "claude-sonnet-4-6"),
        max_tokens=config.get("max_tokens", 1024),
        messages=[{"role": "user", "content": compiled}]
    )
    
    return response.content[0].text


texts = [
    ("Hello, the weather is really nice today!", "Korean"),
    ("The quick brown fox jumps over the lazy dog.", "Korean"),
    ("Bonjour le monde!", "English"),
]

for source, target in texts:
    result = translate(source, target)
    print(f"Source (to {target}): {source}")
    print(f"Translation: {result}")
    print()

langfuse.flush()

## Step 6: Code review with Chat-type prompt

In [ ]:
@observe()
def review_code(code: str, language: str) -> str:
    prompt_template = langfuse.get_prompt("code-reviewer")
    
    # Chat type compile() returns a messages list
    compiled_messages = prompt_template.compile(
        language=language,
        code=code
    )
    
    config = prompt_template.config
    
    response = client.messages.create(
        model=config.get("model", "claude-sonnet-4-6"),
        max_tokens=config.get("max_tokens", 2048),
        system=compiled_messages[0]["content"],
        messages=[compiled_messages[1]]
    )
    
    return response.content[0].text


sample_code = """
def get_user_data(user_id):
    import sqlite3
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query = f"SELECT * FROM users WHERE id = {user_id}"
    cursor.execute(query)
    result = cursor.fetchone()
    return result
"""

review = review_code(sample_code, "Python")
print(review)

langfuse.flush()

## Step 7: Update prompt version

Calling `create_prompt()` again with the same name creates a new version.
Previous versions are preserved as-is.

In [ ]:
# Translation prompt v2 - added format instructions
langfuse.create_prompt(
    name="text-translator",
    prompt="""Please translate the following text into {{target_language}}.
Preserve the nuance and formality level of the original.

Output format:
- Output only the translation
- Do not include any additional explanation or notes

Source text:
{{source_text}}

Translation:""",
    config={
        "model": "claude-sonnet-4-6",
        "max_tokens": 1024,
        "temperature": 0.2  # Lower temperature for more consistent translations
    },
    labels=["production"]  # New version promoted to production
)

print("✅ 'text-translator' v2 created")

# Check the latest production version
latest = langfuse.get_prompt("text-translator")
print(f"Current production version: {latest.version}")

## Step 8: Retrieve a specific version

In [ ]:
# Retrieve version 1 (previous version)
v1_prompt = langfuse.get_prompt("text-translator", version=1)
print(f"v1 temperature: {v1_prompt.config.get('temperature')}")

# Retrieve the latest production version
v2_prompt = langfuse.get_prompt("text-translator", label="production")
print(f"Latest production temperature: {v2_prompt.config.get('temperature')}")

## Summary

| Concept | Description |
|---------|-------------|
| `create_prompt()` | Create a new prompt or a new version |
| `get_prompt(name)` | Retrieve the latest version with the production label |
| `get_prompt(name, version=N)` | Retrieve a specific version |
| `prompt.compile(**kwargs)` | Substitute template variables |
| `prompt.config` | Access stored model settings |
| `labels=["production"]` | Promote to production version |

Next: **04_scoring_and_evaluation.ipynb** → Evaluating LLM output quality